<a href="https://colab.research.google.com/github/RushiKP14/Tensorflow/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-03-08 00:50:58--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.3’

book-crossings.zip. 100%[===================>]  24.88M   141MB/s    in 0.2s    

2025-03-08 00:50:58 (141 MB/s) - ‘book-crossings.zip.3’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [174]:
# add your code here - consider creating a new cell for each section of code
print(df_ratings.head())
df_books.head()
#df_ratings.shape
#df_ratings.describe()

     user        isbn  rating
0  276725  034545104X     0.0
1  276726  0155061224     5.0
2  276727  0446520802     0.0
3  276729  052165615X     3.0
4  276729  0521795028     6.0


,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


# Step 1

In [146]:
# Step 1: Remove users who gave less than 200 ratings
user_counts = df_ratings['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index
df_filtered_users = df_ratings[df_ratings['user'].isin(valid_users)]
#df_filtered_users.reset_index(drop=True)

### Ignore next cell

In [ ]:
list_user = df_ratings.user.value_counts()
print(list_user)
keep_user=[]
for i in list_user.index:
  if list_user.get(i)>=200:
    keep_user.append(i)
#print(keep_user)
flag=1
for user in keep_user:
  mask = df_ratings['user'] == user
  if flag==1:
    df_ratings_new = df_ratings.loc[mask]
    flag=0
  else:
    df_ratings_new = pd.concat([df_ratings_new, df_ratings.loc[mask]], ignore_index=True)
print(df_ratings_new)
sum(list_user.get(keep_user))

# Step 2

In [147]:
# Step 2: Remove ISBNs that got less than 100 ratings
isbn_counts = df_ratings.isbn.value_counts()
valid_isbns = isbn_counts[isbn_counts >= 100].index
df_final = df_filtered_users[df_filtered_users['isbn'].isin(valid_isbns)]
df_final = df_final.reset_index(drop=True)
df_final

,user,isbn,rating
0,277427,002542730X,10.0
1,277427,0060930535,0.0
2,277427,0060934417,0.0
3,277427,0061009059,9.0
4,277427,0140067477,0.0
...,...,...,...
49776,275970,0804111359,0.0
49777,275970,140003065X,0.0
49778,275970,1400031346,0.0
49779,275970,1400031354,0.0


# Ignore next 3 cells

In [ ]:
list_isbn = df_ratings.isbn.value_counts()
print(list_isbn)
keep_isbn=[]
for i in list_isbn.index:
  if list_isbn.get(i)>=100:
    keep_isbn.append(i)
print(keep_isbn)
flag=1
for isbn in keep_isbn:
  mask = df_ratings_new['isbn'] == isbn
  if flag==1:
    df_ratings_final = df_ratings_new.loc[mask]
    flag=0
  else:
    df_ratings_final = pd.concat([df_ratings_final, df_ratings_new.loc[mask]], ignore_index=True)
print(df_ratings_final)
#sum(list_isbn.get(keep_isbn))

In [ ]:
#ratings_neighb.loc[11676,'0971880107'] = df_ratings_final[(df_ratings_final['user']==11676) & (df_ratings_final['isbn']=='0971880107')]['rating']
#ratings_neighb.loc[11676,'0971880107']=df_ratings_final[(df_ratings_final['user']==11676) & (df_ratings_final['isbn']=='0971880107')]['rating'].values[0]
#ratings_neighb
df_ratings_final.loc[df_ratings_final['isbn']=='0971880107', 'rating']
data = np.ones((len(keep_user),len(keep_isbn)))*5
i,j = 0,0
data[i,j] = df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].values[0]
type(df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].ndim)
#print(data)
data.shape
data[905,730]

In [ ]:
#df_ratings_final.loc[df_ratings_final['user']==11676]
data = np.ones((len(keep_user),len(keep_isbn)))*5
#ratings_neighb = pd.DataFrame(index = keep_user, columns = keep_isbn)
for i in range(len(keep_user)):
  for j in range(len(keep_isbn)):
    if df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].empty:
      continue
    else:
      data[i,j] = df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].values[0]
#ratings_neighb['0971880107'] = df_ratings_final.loc[df_ratings_final['isbn']=='0971880107', 'rating']

# From here on, all cells are part of the project.

In [172]:
data = df_final.pivot(index='user', columns='isbn', values='rating')
#mean_column = data.mean()
#print(mean_column)
#final_data = data.fillna(value=mean_column)
final_data = data.fillna(0.0)
final_data = final_data.T
print(final_data)
ratings_numpy = final_data.to_numpy()
#ratings_numpy = mean_column.to_numpy().reshape((-1,1))
#print(ratings_numpy)
#x = mean_column[isbn]

user        254     2276    2766    2977    3363    4017    4385    6242    \
isbn                                                                         
002542730X     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
0060008032     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
0060096195     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
006016848X     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
0060173289     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
...            ...     ...     ...     ...     ...     ...     ...     ...   
1573227331     0.0     0.0     0.0     0.0     0.0     0.0     0.0     6.0   
1573229326     0.0     0.0     0.0     0.0     0.0     0.0     0.0     6.0   
1573229571     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
1592400876     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
1878424319     0.0     0.0     0.0     0.0     0.0     0.0     0

In [158]:
neigh = NearestNeighbors(n_neighbors=6, metric='cosine',algorithm='brute').fit(ratings_numpy)

In [168]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  isbn = df_books[df_books['title']==book]['isbn'].to_numpy()[0]
  x = final_data.loc[isbn].to_numpy()
  distances, indices = neigh.kneighbors([x])
  recommended_isbns = final_data.index[indices[0]]
  recommended_books = [book]
  y=[]
  for i in range(1,6):
    if recommended_isbns[i] in df_books['isbn'].values:
      y.append([df_books.loc[df_books['isbn']==recommended_isbns[i]]['title'].to_numpy()[0], distances[0][i]])
    else:
      continue
  y=y[::-1]
  recommended_books.append(y)
  return recommended_books

In [169]:
books = get_recommends('The Queen of the Damned (Vampire Chronicles (Paperback))')
books

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['Catch 22', 0.7939835],
  ['The Witching Hour (Lives of the Mayfair Witches)', 0.74486566],
  ['Interview with the Vampire', 0.73450685],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.53763384],
  ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.51784116]]]

In [170]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016211], ['The Weight of Water', 0.77085835], ['The Surgeon', 0.7699411], ['I Know This Much Is True', 0.7677075], ['The Lovely Bones: A Novel', 0.7234864]]]
You passed the challenge! 🎉🎉🎉🎉🎉
